In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from pathlib import Path
from harbor.analysis.cross_docking import DockingDataModel
from plotting_params import *
import numpy as np
from rdkit import Chem

In [ ]:
def get_label(var):
    return label_map.get(var,var)

## input files

In [ ]:
analyzed_path = Path("/Users/alexpayne/Scientific_Projects/mers-drug-discovery/sars2-retrospective-analysis/analyzed_results/")

In [ ]:
posit_results = [Path(analyzed_path / csv) for csv in ["x_to_not_x_posit_combined_results.csv", "not_x_to_x_posit_combined_results.csv", "not_x_to_x_posit_5_refs_combined_results.csv"]
                ]

In [ ]:
sdf = pd.concat([pd.read_csv(file) for file in posit_results])
sdf["Error_Lower"] = sdf["Fraction"] - sdf["CI_Lower"]
sdf["Error_Lower"] = sdf["Error_Lower"].apply(lambda x: 0 if x < 0 else x)
sdf["Error_Upper"] = sdf["CI_Upper"] - sdf["Fraction"]
sdf["Error_Upper"] = sdf["Error_Upper"].apply(lambda x: 0 if x < 0 else x)

# replace brackets in the query and ref columns
query = "Query_Scaffold_ID_Subset"
ref = "Reference_Scaffold_ID_Subset"
def process_column(value):
    if isinstance(value, str) and "[" in value:
        mylist = value.strip("[").strip("]").split(",")
        if len(mylist) == 1:
            return str(int(mylist[0]) + 1)
    return np.nan
    
sdf[query] = sdf[query].apply(process_column)
sdf["qint"] = sdf[query].astype(float).tolist()
sdf[ref] = sdf[ref].apply(process_column)
sdf["rint"] = sdf[ref].astype(float).tolist()

In [ ]:
sdf.iloc[2]["Evaluator_Model"]

## output files

In [ ]:
figpath = Path("../figures")
figpath.mkdir(exist_ok=True)

# Plot x to not x

## all structures

In [ ]:
# sort 
df = sdf[(sdf.Scaffold_Split_Option == "x_to_not_x")&(sdf.N_Reference_Structures == 542)]
df = df.sort_values(["qint", "Score"], ascending=[True, False])

jitter = 0.4
mult_factor = 2

# manual jitter to separate by score
jitter_vector = df.Score.apply(lambda x: jitter if x == "POSIT_Probability" else -jitter)
df["qint_jittered"] = mult_factor * df['qint'] + jitter_vector

xvals = list(mult_factor * df['qint'])
linevals = [xval - 1 for xval in xvals[2:]]


fig = plt.figure(figsize=(12,6))
ax = sns.scatterplot(df, x="qint_jittered", y="Fraction", hue="Score", s=200)
ax.set_ylim(-0.05,1.05)

ax.set(xticks=xvals,
      xticklabels=df[query])
# Add error bars using Matplotlib's errorbar
plt.errorbar(x=df["qint_jittered"], y=df['Fraction'], yerr=(df['Error_Lower'], df['Error_Upper']),  
             fmt='none',  # Remove the default connecting line/markers
             capsize=5,  # Adjust the size of the error bar caps
             color='black',  # Set the color of the error bars
             alpha=0.7,  # Adjust the transparency of the error bars
             elinewidth=1 # Adjust the thickness of the error bars
            )

for lineval in linevals:
    # Add vertical line at specific x position
    plt.axvline(x=lineval, color='black', linestyle='--', lw=0.5)

fig = update_labels(fig, label_map, x_label="Query Ligand Scaffold ID")
save_figure(fig, figpath / "success_by_query_ligand_scaffold" )

plt.show()

## n refs

In [ ]:
df = sdf[(sdf.Scaffold_Split_Option == "x_to_not_x")]
# sort 
df = df.sort_values(["qint", "Score"], ascending=[True, False])

jitter = 0.4
mult_factor = 2

# manual jitter to separate by score
jitter_vector = df.Score.apply(lambda x: jitter if x == "POSIT_Probability" else -jitter)
df["qint_jittered"] = mult_factor * df['qint'] + jitter_vector

xvals = list(mult_factor * df['qint'])
linevals = [xval - 1 for xval in xvals[2:]]


fig, ax = plt.subplots(figsize=(12, 6))

# Create log-normalized colormap
from matplotlib.colors import LogNorm
norm = LogNorm(df['N_Reference_Structures'].min(), df['N_Reference_Structures'].max())

# Apply the log-normalized colormap to the scatter plot
scatter = sns.scatterplot(data=df, x="qint_jittered", y="Fraction",
                         hue="N_Reference_Structures", style="Score",
                         palette="crest_r", ax=ax, hue_norm=norm, legend='full')

ax.set(xticks=xvals, xticklabels=df[query])
ax.set_ylim(-0.05,1.05)

for lineval in linevals:
    ax.axvline(x=lineval, color='black', linestyle='--', lw=0.5)

ax.set_xlabel(get_label("Query Ligand Scaffold ID"), fontsize=FONT_SIZES["xlabel"], fontweight="bold")
ax.set_ylabel(get_label(Y_LABEL), fontsize=FONT_SIZES["ylabel"], fontweight="bold")

plt.yticks(fontsize=FONT_SIZES["ticks"])
plt.xticks(fontsize=FONT_SIZES["ticks"])

# Move legend outside the plot and store the new legend
legend = ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
texts = legend.get_texts()

# Update legend text with mapped values
legend_subtitles = ["N_Reference_Structures", "Score"]
for text in texts:
    if text._text in legend_subtitles:
        plt.setp(text, fontsize=FONT_SIZES["legend_title"], fontweight="bold")
    text.set_text(get_label(text._text))

plt.setp(legend.get_title(), fontsize=FONT_SIZES["legend_title"], fontweight="bold")
plt.setp(legend.get_texts(), fontsize=FONT_SIZES["legend_text"])

save_figure(fig, figpath / "success_by_query_ligand_scaffold_all_refs")
plt.show()

## just 5 refs

In [ ]:
# sort 
df = sdf[(sdf.Scaffold_Split_Option == "x_to_not_x")&(sdf.N_Reference_Structures == 5)]
df = df.sort_values(["qint", "Score"], ascending=[True, False])

jitter = 0.4
mult_factor = 2

# manual jitter to separate by score
jitter_vector = df.Score.apply(lambda x: jitter if x == "POSIT_Probability" else -jitter)
df["qint_jittered"] = mult_factor * df['qint'] + jitter_vector

xvals = list(mult_factor * df['qint'])
linevals = [xval - 1 for xval in xvals[2:]]


fig = plt.figure(figsize=(12,6))
ax = sns.scatterplot(df, x="qint_jittered", y="Fraction", hue="Score", s=200)
ax.set_ylim(-0.05,1.05)

ax.set(xticks=xvals,
      xticklabels=df[query])
# Add error bars using Matplotlib's errorbar
plt.errorbar(x=df["qint_jittered"], y=df['Fraction'], yerr=(df['Error_Lower'], df['Error_Upper']),  
             fmt='none',  # Remove the default connecting line/markers
             capsize=5,  # Adjust the size of the error bar caps
             color='black',  # Set the color of the error bars
             alpha=0.7,  # Adjust the transparency of the error bars
             elinewidth=1 # Adjust the thickness of the error bars
            )

for lineval in linevals:
    # Add vertical line at specific x position
    plt.axvline(x=lineval, color='black', linestyle='--', lw=0.5)

fig = update_labels(fig, label_map, x_label="Query Ligand Scaffold ID")
save_figure(fig, figpath / "success_by_query_ligand_scaffold_5_refs" )

plt.show()

# Plot not x to x

## all refs

In [ ]:
# sort 
df = sdf[(sdf.Scaffold_Split_Option == "not_x_to_x")&(sdf.N_Reference_Structures != 5)]
df["N_Reference_Structures"] = df.N_Reference_Structures.apply(lambda x: x if x == 5 else "All")
df = df.sort_values(["rint", "Score"], ascending=[True, False])

jitter = 0.4
mult_factor = 2

# manual jitter to separate by score
jitter_vector = df.Score.apply(lambda x: jitter if x == "POSIT_Probability" else -jitter)
df["rint_jittered"] = mult_factor * df['rint'] + jitter_vector

xvals = list(mult_factor * df['rint'])
# linevals = [xval - 1.25 for xval in xvals + [xvals[-1] + mult_factor]]
linevals = [xval - 1 for xval in xvals[2:]]


fig = plt.figure(figsize=(12,6))
g = sns.scatterplot(df, x="rint_jittered", y="Fraction", hue="Score", s=200)
g.set_ylim(-0.05,1.05)

g.set(xticks=xvals,
      xticklabels=df[ref])
# Add error bars using Matplotlib's errorbar
plt.errorbar(x=df["rint_jittered"], y=df['Fraction'], yerr=(df['Error_Lower'], df['Error_Upper']),  
             fmt='none',  # Remove the default connecting line/markers
             capsize=5,  # Adjust the size of the error bar caps
             color='black',  # Set the color of the error bars
             alpha=0.7,  # Adjust the transparency of the error bars
             elinewidth=1 # Adjust the thickness of the error bars
            )

for lineval in linevals:
    # Add vertical line at specific x position
    plt.axvline(x=lineval, color='black', linestyle='--', lw=0.5)

fig = update_labels(fig, label_map, x_label="Reference Ligand Scaffold ID")
save_figure(fig, figpath / "success_by_ref_ligand_scaffold" )

## with n refs

In [ ]:
df = sdf[(sdf.Scaffold_Split_Option == "not_x_to_x")]
df["N_Reference_Structures"] = df.N_Reference_Structures.apply(lambda x: x if x == 5 else "All")
df = df.sort_values(["rint", "Score"], ascending=[True, False])

jitter = 0.4
mult_factor = 2

# manual jitter to separate by score
jitter_vector = df.Score.apply(lambda x: jitter if x == "POSIT_Probability" else -jitter)
df["rint_jittered"] = mult_factor * df['rint'] + jitter_vector

xvals = list(mult_factor * df['rint'])
linevals = [xval - 1 for xval in xvals[2:]]


fig, ax = plt.subplots(figsize=(12, 6))

scatter = sns.scatterplot(data=df, x="rint_jittered", y="Fraction",
                         hue="N_Reference_Structures", style="Score",
                         ax=ax)
ax.set_ylim(-0.05,1.05)

ax.set(xticks=xvals, xticklabels=df[ref])
# Add error bars using Matplotlib's errorbar
plt.errorbar(x=df["rint_jittered"], y=df['Fraction'], yerr=(df['Error_Lower'], df['Error_Upper']),  
             fmt='none',  # Remove the default connecting line/markers
             capsize=5,  # Adjust the size of the error bar caps
             color='black',  # Set the color of the error bars
             alpha=0.7,  # Adjust the transparency of the error bars
             elinewidth=1 # Adjust the thickness of the error bars
            )

for lineval in linevals:
    ax.axvline(x=lineval, color='black', linestyle='--', lw=0.5)

ax.set_xlabel(get_label("Reference Ligand Scaffold ID"), fontsize=FONT_SIZES["xlabel"], fontweight="bold")
ax.set_ylabel(get_label(Y_LABEL), fontsize=FONT_SIZES["ylabel"], fontweight="bold")

plt.yticks(fontsize=FONT_SIZES["ticks"])
plt.xticks(fontsize=FONT_SIZES["ticks"])

# Move legend outside the plot and store the new legend
legend = ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
texts = legend.get_texts()

# Update legend text with mapped values
legend_subtitles = ["N_Reference_Structures", "Score"]
for text in texts:
    if text._text in legend_subtitles:
        plt.setp(text, fontsize=FONT_SIZES["legend_title"], fontweight="bold")
    text.set_text(get_label(text._text))

plt.setp(legend.get_title(), fontsize=FONT_SIZES["legend_title"], fontweight="bold")
plt.setp(legend.get_texts(), fontsize=FONT_SIZES["legend_text"])

save_figure(fig, figpath / "success_by_reference_ligand_scaffold_all_refs")

## with just 5 refs

In [ ]:
# sort 
df = sdf[(sdf.Scaffold_Split_Option == "not_x_to_x")&(sdf.N_Reference_Structures == 5)]
df["N_Reference_Structures"] = df.N_Reference_Structures.apply(lambda x: x if x == 5 else "All")
df = df.sort_values(["rint", "Score"], ascending=[True, False])

jitter = 0.4
mult_factor = 2

# manual jitter to separate by score
jitter_vector = df.Score.apply(lambda x: jitter if x == "POSIT_Probability" else -jitter)
df["rint_jittered"] = mult_factor * df['rint'] + jitter_vector

xvals = list(mult_factor * df['rint'])
# linevals = [xval - 1.25 for xval in xvals + [xvals[-1] + mult_factor]]
linevals = [xval - 1 for xval in xvals[2:]]


fig = plt.figure(figsize=(12,6))
g = sns.scatterplot(df, x="rint_jittered", y="Fraction", hue="Score", s=200)
g.set_ylim(-0.05,1.05)

g.set(xticks=xvals,
      xticklabels=df[ref])
# Add error bars using Matplotlib's errorbar
plt.errorbar(x=df["rint_jittered"], y=df['Fraction'], yerr=(df['Error_Lower'], df['Error_Upper']),  
             fmt='none',  # Remove the default connecting line/markers
             capsize=5,  # Adjust the size of the error bar caps
             color='black',  # Set the color of the error bars
             alpha=0.7,  # Adjust the transparency of the error bars
             elinewidth=1 # Adjust the thickness of the error bars
            )

for lineval in linevals:
    # Add vertical line at specific x position
    plt.axvline(x=lineval, color='black', linestyle='--', lw=0.5)

fig = update_labels(fig, label_map, x_label="Reference Ligand Scaffold ID")
save_figure(fig, figpath / "success_by_ref_ligand_scaffold_5_refs" )

# Plot performance vs # atoms

In [ ]:
posit_raw_df = pd.read_parquet("/Users/alexpayne/Scientific_Projects/mers-drug-discovery/sars2-retrospective-analysis/ALL_combined_results.parquet")

In [ ]:
size_dict = {}
names = []
for i in range(20):
    scaffold_smiles = posit_raw_df[posit_raw_df.cluster_id == i].groupby(["Query_Ligand"]).head(1).scaffold_smarts.unique()[0]
    mol = Chem.MolFromSmiles(scaffold_smiles)
    names.append(f"Scaffold_{i+1}")
    size_dict[i+1] = mol.GetNumHeavyAtoms()

## 5 refs

In [ ]:
df = sdf[sdf["N_Reference_Structures"] == 5]

In [ ]:
df["combined_scaffold"] = df.rint.fillna(0) + df.qint.fillna(0)
df["css"] = df.combined_scaffold.astype(int).astype(str)

In [ ]:
df["N_Atoms_In_Scaffold"] = df["combined_scaffold"].apply(lambda x: int(size_dict.get(x)))

In [ ]:
df.sort_values(["combined_scaffold"], inplace=True)
x_label = "Number of Atoms in Ligand Scaffold"
# df[x_label] = df.N_Atoms_In_Scaffold.astype(str)
df[x_label] = df.N_Atoms_In_Scaffold

In [ ]:
fig = plt.figure(figsize=(9,6))
# df = df[(df["Score"] == "POSIT_Probability")&(df["Scaffold_Split_Option"] == "x_to_not_x")]
df = df[(df["Score"] == "POSIT_Probability")]
# ax = sns.scatterplot(df, x=x_label, y="Fraction", hue="Score", style="Scaffold_Split_Option")
ax = sns.scatterplot(df, x=x_label, y="Fraction", hue="Scaffold_Split_Option")
ax.set_ylim((-0.1,1))
label_map["x_to_not_x"] = "Scaffold as Query"
label_map["not_x_to_x"] = "Scaffold as Refrence"
label_map["Scaffold_Split_Option"] = ""
# Move legend outside the plot and store the new legend
legend = ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
texts = legend.get_texts()
xvals = range(5,27)
ax.set(xticks=xvals, xticklabels=xvals)

ax.set_xlabel(x_label, fontsize=FONT_SIZES["xlabel"], fontweight="bold")
ax.set_ylabel(get_label(Y_LABEL), fontsize=FONT_SIZES["ylabel"], fontweight="bold")

plt.yticks(fontsize=FONT_SIZES["ticks"])
plt.xticks(fontsize=FONT_SIZES["ticks"])

# Move legend outside the plot and store the new legend
legend = ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
texts = legend.get_texts()

# Update legend text with mapped values
legend_subtitles = ["N_Reference_Structures", "Score"]
for text in texts:
    if text._text in legend_subtitles:
        plt.setp(text, fontsize=FONT_SIZES["legend_title"], fontweight="bold")
    text.set_text(get_label(text._text))

plt.setp(legend.get_title(), fontsize=FONT_SIZES["legend_title"], fontweight="bold")
plt.setp(legend.get_texts(), fontsize=FONT_SIZES["legend_text"])

save_figure(fig, figpath / "success_vs_n_atoms_in_scaffold" )

In [ ]:
fig = plt.figure(figsize=(9,6))
# df = df[(df["Score"] == "POSIT_Probability")&(df["Scaffold_Split_Option"] == "x_to_not_x")]
df = df[(df["Score"] == "POSIT_Probability")]
# ax = sns.scatterplot(df, x=x_label, y="Fraction", hue="Score", style="Scaffold_Split_Option")
ax = sns.scatterplot(df, x=x_label, y="Fraction", hue='css', style="Scaffold_Split_Option", s=300, alpha=0.9)
ax.set_ylim((-0.1,1))
label_map["x_to_not_x"] = "Scaffold as Query"
label_map["not_x_to_x"] = "Scaffold as Refrence"
label_map["Scaffold_Split_Option"] = ""
label_map["css"] = "Scaffold ID"
xvals = range(5,27,5)
ax.set(xticks=xvals, xticklabels=xvals)

ax.set_xlabel(x_label, fontsize=FONT_SIZES["xlabel"], fontweight="bold")
ax.set_ylabel(get_label(Y_LABEL), fontsize=FONT_SIZES["ylabel"], fontweight="bold")

plt.yticks(fontsize=FONT_SIZES["ticks"])
plt.xticks(fontsize=FONT_SIZES["ticks"])

# Add annotations for each point
for i, point in df.iterrows():
    if point["Scaffold_Split_Option"] == "x_to_not_x":
        # Add the css value as text at each point's position
        ax.text(point[x_label], point["Fraction"], 
                str(point['css']),  # The text to display (css value)
                fontsize=9,         # Font size of annotation
                ha='center',        # Horizontal alignment
                va='center',        # Vertical alignment
                color='white',      # Text color for better contrast against colored dots
                fontweight='bold')  # Make text bold for better visibility

# Move legend outside the plot and store the new legend
legend = ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
texts = legend.get_texts()

# Update legend text with mapped values
legend_subtitles = ["Scaffold_Split_Option", "css"]
for text in texts:
    if text._text in legend_subtitles:
        plt.setp(text, fontsize=FONT_SIZES["legend_text"], fontweight="bold")
    if text._text in ["x_to_not_x", "not_x_to_x"]:
        plt.setp(text, fontsize=FONT_SIZES["legend_text"])
    text.set_text(get_label(text._text))

# plt.setp(legend.get_title(), fontsize=FONT_SIZES["legend_title"], fontweight="bold")
# plt.setp(legend.get_texts(), fontsize=FONT_SIZES["legend_text"])

save_figure(fig, figpath / "success_vs_n_atoms_in_scaffold_5_refs_v2" )

## all refs

In [ ]:
sdf[sdf["Scaffold_Split_Option"] == "not_x_to_x"]

In [ ]:
sdf[sdf["N_Reference_Structures"].isna()].nunique()

In [ ]:
df = sdf[(sdf["N_Reference_Structures"] == 542)|(sdf["N_Reference_Structures"].isna())]

In [ ]:
df.nunique()

In [ ]:
df["combined_scaffold"] = df.rint.fillna(0) + df.qint.fillna(0)
df["css"] = df.combined_scaffold.astype(int).astype(str)

In [ ]:
df["N_Atoms_In_Scaffold"] = df["combined_scaffold"].apply(lambda x: int(size_dict.get(x)))

In [ ]:
df.sort_values(["combined_scaffold"], inplace=True)
x_label = "Number of Atoms in Ligand Scaffold"
# df[x_label] = df.N_Atoms_In_Scaffold.astype(str)
df[x_label] = df.N_Atoms_In_Scaffold

In [ ]:
fig = plt.figure(figsize=(9,6))
# df = df[(df["Score"] == "POSIT_Probability")&(df["Scaffold_Split_Option"] == "x_to_not_x")]
df = df[(df["Score"] == "POSIT_Probability")]
# ax = sns.scatterplot(df, x=x_label, y="Fraction", hue="Score", style="Scaffold_Split_Option")
ax = sns.scatterplot(df, x=x_label, y="Fraction", hue="Scaffold_Split_Option")
ax.set_ylim((-0.1,1))
label_map["x_to_not_x"] = "Scaffold as Query"
label_map["not_x_to_x"] = "Scaffold as Refrence"
label_map["Scaffold_Split_Option"] = ""
# Move legend outside the plot and store the new legend
legend = ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
texts = legend.get_texts()
xvals = range(5,27)
ax.set(xticks=xvals, xticklabels=xvals)

ax.set_xlabel(x_label, fontsize=FONT_SIZES["xlabel"], fontweight="bold")
ax.set_ylabel(get_label(Y_LABEL), fontsize=FONT_SIZES["ylabel"], fontweight="bold")

plt.yticks(fontsize=FONT_SIZES["ticks"])
plt.xticks(fontsize=FONT_SIZES["ticks"])

# Move legend outside the plot and store the new legend
legend = ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
texts = legend.get_texts()

# Update legend text with mapped values
legend_subtitles = ["N_Reference_Structures", "Score"]
for text in texts:
    if text._text in legend_subtitles:
        plt.setp(text, fontsize=FONT_SIZES["legend_title"], fontweight="bold")
    text.set_text(get_label(text._text))

plt.setp(legend.get_title(), fontsize=FONT_SIZES["legend_title"], fontweight="bold")
plt.setp(legend.get_texts(), fontsize=FONT_SIZES["legend_text"])

save_figure(fig, figpath / "success_vs_n_atoms_in_scaffold" )

In [ ]:
fig = plt.figure(figsize=(9,6))
df.sort_values("Scaffold_Split_Option", inplace=True, ascending=False)
# df = df[(df["Score"] == "POSIT_Probability")&(df["Scaffold_Split_Option"] == "x_to_not_x")]
df = df[(df["Score"] == "POSIT_Probability")]
# ax = sns.scatterplot(df, x=x_label, y="Fraction", hue="Score", style="Scaffold_Split_Option")
ax = sns.scatterplot(df, x=x_label, y="Fraction", hue='css', style="Scaffold_Split_Option", s=400, alpha=0.9)
ax.set_ylim((-0.05,1.05))
label_map["x_to_not_x"] = "Scaffold as Query"
label_map["not_x_to_x"] = "Scaffold as Refrence"
label_map["Scaffold_Split_Option"] = ""
label_map["css"] = "Scaffold ID"
xvals = range(5,27,5)
ax.set(xticks=xvals, xticklabels=xvals)

ax.set_xlabel(x_label, fontsize=FONT_SIZES["xlabel"], fontweight="bold")
ax.set_ylabel(get_label(Y_LABEL), fontsize=FONT_SIZES["ylabel"], fontweight="bold")

plt.yticks(fontsize=FONT_SIZES["ticks"])
plt.xticks(fontsize=FONT_SIZES["ticks"])

# Add annotations for each point
for i, point in df.iterrows():
    if point["Scaffold_Split_Option"] == "x_to_not_x":
        # Add the css value as text at each point's position
        ax.text(point[x_label], point["Fraction"], 
                str(point['css']),  # The text to display (css value)
                fontsize=12,         # Font size of annotation
                ha='center',        # Horizontal alignment
                va='center',        # Vertical alignment
                color='white',      # Text color for better contrast against colored dots
                fontweight='bold')  # Make text bold for better visibility

# Move legend outside the plot and store the new legend
legend = ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
texts = legend.get_texts()

# Update legend text with mapped values
legend_subtitles = ["Scaffold_Split_Option", "css"]
for text in texts:
    if text._text in legend_subtitles:
        plt.setp(text, fontsize=FONT_SIZES["legend_text"], fontweight="bold")
    if text._text in ["x_to_not_x", "not_x_to_x"]:
        plt.setp(text, fontsize=FONT_SIZES["legend_text"])
    text.set_text(get_label(text._text))

# plt.setp(legend.get_title(), fontsize=FONT_SIZES["legend_title"], fontweight="bold")
# plt.setp(legend.get_texts(), fontsize=FONT_SIZES["legend_text"])

save_figure(fig, figpath / "success_vs_n_atoms_in_scaffold_v2" )

# what if I just sort them by atom count?

In [ ]:
posit_raw_df = pd.read_parquet("/Users/alexpayne/Scientific_Projects/mers-drug-discovery/sars2-retrospective-analysis/ALL_combined_results.parquet")

In [ ]:
size_dict = {}
names = []
for i in range(20):
    scaffold_smiles = posit_raw_df[posit_raw_df.cluster_id == i].groupby(["Query_Ligand"]).head(1).scaffold_smarts.unique()[0]
    mol = Chem.MolFromSmiles(scaffold_smiles)
    names.append(f"Scaffold_{i+1}")
    size_dict[i+1] = mol.GetNumHeavyAtoms()

In [ ]:
from rdkit import Chem
df = sdf[(sdf["N_Reference_Structures"] == 5)&(sdf["Score"] == "POSIT_Probability")]

In [ ]:
df["combined_scaffold"] = df.rint.fillna(0) + df.qint.fillna(0)
df["css"] = df.combined_scaffold.astype(int).astype(str)

In [ ]:
df["N_Atoms_In_Scaffold"] = df["combined_scaffold"].apply(lambda x: int(size_dict.get(x)))

In [ ]:
df.sort_values(["combined_scaffold"], inplace=True)
x_label = "Number of Atoms in Ligand Scaffold"
# df[x_label] = df.N_Atoms_In_Scaffold.astype(str)
df[x_label] = df.N_Atoms_In_Scaffold

In [ ]:
df.sort_values(["N_Atoms_In_Scaffold"], inplace=True)
jitter = 0.4
mult_factor = 2

In [ ]:
df

In [ ]:
# manual jitter to separate by score
# jitter_vector = df.Scaffold_Split_Option.apply(lambda x: jitter if x == "x_to_not_x" else -jitter)
# df["combined_scaffold_jittered"] = mult_factor * df['combined_scaffold'] + jitter_vector

In [ ]:
import matplotlib as mpl

fig = plt.figure(figsize=(12, 6))

# Create the scatterplot with hue and style
scatter = sns.scatterplot(
    data=df,
    x="css",
    y="Fraction",
    style="Scaffold_Split_Option",  # Keep legend for this variable
    hue="N_Atoms_In_Scaffold",  # Use colorbar for this variable
    s=200,
    palette="viridis",  # Choose a colormap
    legend="brief"  # Keep legend for style
)

# Add error bars
plt.errorbar(
    x=df["css"],
    y=df["Fraction"],
    yerr=(df["Error_Lower"], df["Error_Upper"]),
    fmt="none",  # Remove the default connecting line/markers
    capsize=5,  # Adjust the size of the error bar caps
    color="black",  # Set the color of the error bars
    alpha=0.7,  # Adjust the transparency of the error bars
    elinewidth=1,  # Adjust the thickness of the error bars
)

# Create a colorbar for the hue variable
norm = mpl.colors.Normalize(vmin=df["N_Atoms_In_Scaffold"].min(), vmax=df["N_Atoms_In_Scaffold"].max())
sm = mpl.cm.ScalarMappable(cmap="viridis", norm=norm)
sm.set_array([])  # Required for ScalarMappable
cbar = plt.colorbar(sm, ax=plt.gca(), label="Number of Atoms in Scaffold")

# Update labels
fig = update_labels(fig, label_map, x_label="Reference Ligand Scaffold ID")

# Test if results make sense

In [ ]:
posit_raw = DockingDataModel.deserialize("/Users/alexpayne/Scientific_Projects/mers-drug-discovery/sars2-retrospective-analysis/ALL_combined_results.parquet")

In [ ]:
import harbor.analysis.cross_docking as cd
from importlib import reload
reload(cd)

In [ ]:
ev1 = cd.Evaluator.from_json_str(sdf[(sdf["rint"] == 13)&(sdf["N_Reference_Structures"]==200)&(sdf["Score"]=="RMSD")]["Evaluator_Model"].tolist()[0])
ev1.n_bootstraps=10
data1 = ev1.run_pose_selector([posit_raw])
data1 = ev1.run_similarity_split(data1)
data1 = ev1.run_dataset_split(data1)

In [ ]:
ev2 = Evaluator.from_json_str(sdf[(sdf["rint"] == 13)&(sdf["N_Reference_Structures"]==542)&(sdf["Score"]=="RMSD")]["Evaluator_Model"].tolist()[0])
ev2.n_bootstraps=10
data2 = ev2.run_pose_selector(posit_raw)
data2 = ev1.run_similarity_split(data2)
data2 = ev2.run_dataset_split(data2)

In [ ]:
data1[0].dataframe.nunique()

In [ ]:
data2[0].dataframe.nunique()

In [ ]:
evs = pd.read_csv("/Users/alexpayne/Scientific_Projects/mers-drug-discovery/sars2-retrospective-analysis/x_to_y_scaffold_split_5_refs_evaluators.csv

In [ ]:
evs.nunique()

In [ ]:
evs.groupby("Score").nunique()

In [ ]:
len(evs.groupby(["Score", "Query_Scaffold_ID_Subset", "N_Reference_Structures"]).nunique())

In [ ]:
evs[evs["N_Reference_Structures"] != 5]

In [ ]:
reload(cd)
# update default settings
default = cd.EvaluatorFactory(name="default")
default.scorer_settings.rmsd_scorer_settings.use = True
default.scorer_settings.posit_scorer_settings.use = True

# Scaffold split options
default_scaffold = default.__deepcopy__()
default_scaffold.name = "default_scaffold_settings"
default_scaffold.dataset_before_similarity = False
default_scaffold.combine_reference_and_similarity_splits = True
default_scaffold.pairwise_split_settings.use = True
default_scaffold.pairwise_split_settings.scaffold_split_settings.use = True
default_scaffold.pairwise_split_settings.scaffold_split_settings.reference_scaffold_min_count = (
    5
)
default_scaffold.pairwise_split_settings.scaffold_split_settings.query_scaffold_min_count = (
    5
)
# x_to_y default
x_to_y_default = default_scaffold.__deepcopy__()
x_to_y_default.name = "x_to_y_scaffold_split"
x_to_y_default.pairwise_split_settings.scaffold_split_settings.scaffold_split_option = (
    cd.ScaffoldSplitOptions.X_TO_Y
)
x_to_y_default.to_yaml_file(Path("./"))

# x to y scaffold split with 5 refs
evf = x_to_y_default.__deepcopy__()
evf.name = "x_to_y_scaffold_split_5_refs"
evf.reference_split_settings.use = True
evf.reference_split_settings.random_split_settings.use = True
evf.reference_split_settings.n_reference_structures = [5]
evf.to_yaml_file(Path("./"))

In [ ]:
evs = evf.create_evaluators(posit_raw)

In [ ]:
len(evs)

In [ ]:
evf.create_reference_splits()

In [ ]:
def get_valid_scaffolds(dataframe, column, min_count):
    if min_count:
        return (
            dataframe[column]
            .value_counts()[lambda counts: counts >= min_count]
            .index.tolist()
        )
    return []

In [ ]:
import itertools
scaffold_settings = evf.pairwise_split_settings.scaffold_split_settings
query_scaffolds = get_valid_scaffolds(
                    posit_raw.get_lig_dataframe(),
                    scaffold_settings.query_scaffold_id_column,
                    evf.pairwise_split_settings.scaffold_split_settings.query_scaffold_min_count,
                )
ref_scaffolds = get_valid_scaffolds(
                    posit_raw.get_ref_dataframe(),
                    evf.pairwise_split_settings.scaffold_split_settings.reference_scaffold_id_column,
                    evf.pairwise_split_settings.scaffold_split_settings.reference_scaffold_min_count,
                )
scaffolds = set(query_scaffolds).union(ref_scaffolds)
split_option = cd.ScaffoldSplitOptions.X_TO_Y
scaffold_list = [[s] for s in scaffolds]
not_x_scaffold_list = [
    [r for r in scaffolds if r != q] for q in scaffolds
]
all_scaffold_list = [scaffolds for _ in scaffolds]

query_scaffolds = scaffold_list
ref_scaffolds = scaffold_list

if split_option == cd.ScaffoldSplitOptions.X_TO_X:
    pass

if split_option == cd.ScaffoldSplitOptions.X_TO_Y:
    query_scaffolds, ref_scaffolds = zip(
        *[
            (q, r)
            for q, r in itertools.product(scaffold_list, scaffold_list)
            if q != r
        ]
    )

In [ ]:
len(query_scaffolds)

In [ ]:
len(ref_scaffolds)

In [ ]:
splits = [cd. ScaffoldSplit(
    query_scaffold_id_column=scaffold_settings.query_scaffold_id_column,
    reference_scaffold_id_column=scaffold_settings.reference_scaffold_id_column,
    query_scaffold_id_subset=query,
    reference_scaffold_id_subset=ref,
    split_option=split_option,
)
for query, ref in zip(query_scaffolds, ref_scaffolds)
]

In [ ]:
len(splits)

In [ ]:
# Scaffold split options
default_scaffold = default.__deepcopy__()
default_scaffold.name = "default_scaffold_settings"
default_scaffold.pairwise_split_settings.use = True
default_scaffold.pairwise_split_settings.scaffold_split_settings.use = True
default_scaffold.pairwise_split_settings.scaffold_split_settings.reference_scaffold_min_count = (
    5
)
default_scaffold.pairwise_split_settings.scaffold_split_settings.query_scaffold_min_count = (
    5
)

# x_to_x default
x_to_x_default = default_scaffold.__deepcopy__()
x_to_x_default.name = "x_to_x_scaffold_split"
x_to_x_default.pairwise_split_settings.scaffold_split_settings.scaffold_split_option = (
    cd.ScaffoldSplitOptions.X_TO_X
)
x_to_x_default.to_yaml_file(Path("./"))

In [ ]:
x_to_x_default.create_evaluators(posit_raw)

In [ ]:
xd = x_to_x_default.create_pairwise_split(posit_raw)

In [ ]:
len(xd)

In [ ]:
# x_to_x 5 refs
evf = x_to_x_default.__deepcopy__()
evf.name = "x_to_x_scaffold_split_5_refs"
evf.pairwise_split_settings.scaffold_split_settings.scaffold_split_option = (
    cd.ScaffoldSplitOptions.X_TO_X
)
evf.dataset_before_similarity = False
evf.combine_reference_and_similarity_splits = True
evf.reference_split_settings.use = True
evf.reference_split_settings.random_split_settings.use = True
evf.reference_split_settings.n_reference_structures = [5]
evf.to_yaml_file(Path("./"))